In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Athletes.csv,Athletes.csv,418514,1781106169000
dbfs:/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Coaches.csv,Coaches.csv,16893,1781106168000
dbfs:/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/EntriesGender.csv,EntriesGender.csv,1126,1781106168000
dbfs:/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Medals.csv,Medals.csv,2414,1781106169000
dbfs:/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Teams.csv,Teams.csv,35303,1781106172000


In [0]:
athletes_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Athletes.csv"

coaches_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Coaches.csv"

medals_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Medals.csv"

teams_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/Teams.csv"

entriesGender_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Raw Data/EntriesGender.csv"

In [0]:
athletes = spark.read.format('csv').option("header","true").option("inferSchema","true").load(athletes_path)

coaches = spark.read.format('csv').option("header","true").option("inferSchema","true").load(coaches_path)

medals = spark.read.format('csv').option("header","true").option("inferSchema","true").load(medals_path)

teams = spark.read.format('csv').option("header","true").option("inferSchema","true").load(teams_path)

entriesGender = spark.read.format('csv').option("header","true").option("inferSchema","true").load(entriesGender_path)


In [0]:
athletes.show()


+--------------------+--------------------+-------------------+
|                Name|                 NOC|         Discipline|
+--------------------+--------------------+-------------------+
|     AALERUD Katrine|              Norway|       Cycling Road|
|         ABAD Nestor|               Spain|Artistic Gymnastics|
|   ABAGNALE Giovanni|               Italy|             Rowing|
|      ABALDE Alberto|               Spain|         Basketball|
|       ABALDE Tamara|               Spain|         Basketball|
|           ABALO Luc|              France|           Handball|
|        ABAROA Cesar|               Chile|             Rowing|
|       ABASS Abobakr|               Sudan|           Swimming|
|    ABBASALI Hamideh|Islamic Republic ...|             Karate|
|       ABBASOV Islam|          Azerbaijan|          Wrestling|
|        ABBINGH Lois|         Netherlands|           Handball|
|         ABBOT Emily|           Australia|Rhythmic Gymnastics|
|       ABBOTT Monica|United States of .

# **_Schema_**

In [0]:
print("Athletes: ")
athletes.printSchema()

print("Coaches: ")
coaches.printSchema()

print("Medals: ")
medals.printSchema()

print("Teams: ")
teams.printSchema()

print("EntriesGender: ")
entriesGender.printSchema()

Athletes: 
root
 |-- Name: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Discipline: string (nullable = true)

Coaches: 
root
 |-- Name: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)

Medals: 
root
 |-- Rank: integer (nullable = true)
 |-- Team/NOC: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)

Teams: 
root
 |-- Name: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Event: string (nullable = true)

EntriesGender: 
root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



### **NULLS**

In [0]:
from pyspark.sql.functions import col,isnan,when,count

print("Athletes: ")
athletes.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in athletes.columns
]).show()

print("Coaches: ")
coaches.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in coaches.columns
]).show()

print("Medals: ")
medals.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in medals.columns
]).show()

print("Teams: ")
teams.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in teams.columns
]).show()

print("EntriesGender: ")
entriesGender.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in entriesGender.columns
]).show()

Athletes: 
+----+---+----------+
|Name|NOC|Discipline|
+----+---+----------+
|   0|  0|         0|
+----+---+----------+

Coaches: 
+----+---+----------+-----+
|Name|NOC|Discipline|Event|
+----+---+----------+-----+
|   0|  0|         0|  145|
+----+---+----------+-----+

Medals: 
+----+--------+----+------+------+-----+-------------+
|Rank|Team/NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------+----+------+------+-----+-------------+
|   0|       0|   0|     0|     0|    0|            0|
+----+--------+----+------+------+-----+-------------+

Teams: 
+----+----------+---+-----+
|Name|Discipline|NOC|Event|
+----+----------+---+-----+
|   0|         0|  0|    0|
+----+----------+---+-----+

EntriesGender: 
+----------+------+----+-----+
|Discipline|Female|Male|Total|
+----------+------+----+-----+
|         0|     0|   0|    0|
+----------+------+----+-----+



### **Duplicates**

In [0]:
print("Total Rows in Athletes: ",athletes.count())
print("Distinct Rows in Athletes: ",athletes.distinct().count())
print("Duplicates Rows in Athletes: ",athletes.count()-athletes.distinct().count(), end="\n\n")

print("Total Rows in Coaches: ",coaches.count())
print("Distinct Rows in Coaches: ",coaches.distinct().count())
print("Duplicates Rows in Coaches: ",coaches.count()-coaches.distinct().count(), end="\n\n")

print("Total Rows in Medals: ",medals.count())
print("Distinct Rows in Medals: ",medals.distinct().count())
print("Duplicates Rows in Medals: ",medals.count()-medals.distinct().count(), end="\n\n")

print("Total Rows in Teams: ",teams.count())
print("Distinct Rows in Teams: ",teams.distinct().count())
print("Duplicates Rows in Teams: ",teams.count()-teams.distinct().count(), end="\n\n")

print("Total Rows in EntriesGender: ",entriesGender.count())
print("Distinct Rows in EntriesGender: ",entriesGender.distinct().count())
print("Duplicates Rows in EntriesGender: ",entriesGender.count()-entriesGender.distinct().count())

Total Rows in Athletes:  11085
Distinct Rows in Athletes:  11084
Duplicates Rows in Athletes:  1

Total Rows in Coaches:  394
Distinct Rows in Coaches:  393
Duplicates Rows in Coaches:  1

Total Rows in Medals:  93
Distinct Rows in Medals:  93
Duplicates Rows in Medals:  0

Total Rows in Teams:  743
Distinct Rows in Teams:  743
Duplicates Rows in Teams:  0

Total Rows in EntriesGender:  46
Distinct Rows in EntriesGender:  46
Duplicates Rows in EntriesGender:  0


In [0]:
athletes_clean = athletes.dropDuplicates()
coaches_clean = coaches.dropDuplicates()

### **Adding the Average columns in EntriesGender csv**

In [0]:
from pyspark.sql.functions import round

entriesGender_clean = entriesGender.withColumn(
    "Avg_Female", round(col('Female')/col('Total'), 2)
).withColumn(
    "Avg_Male", round(col('Male')/col('Total'),2)
)

entriesGender_clean.show()

+--------------------+------+----+-----+----------+--------+
|          Discipline|Female|Male|Total|Avg_Female|Avg_Male|
+--------------------+------+----+-----+----------+--------+
|      3x3 Basketball|    32|  32|   64|       0.5|     0.5|
|             Archery|    64|  64|  128|       0.5|     0.5|
| Artistic Gymnastics|    98|  98|  196|       0.5|     0.5|
|   Artistic Swimming|   105|   0|  105|       1.0|     0.0|
|           Athletics|   969|1072| 2041|      0.47|    0.53|
|           Badminton|    86|  87|  173|       0.5|     0.5|
|   Baseball/Softball|    90| 144|  234|      0.38|    0.62|
|          Basketball|   144| 144|  288|       0.5|     0.5|
|    Beach Volleyball|    48|  48|   96|       0.5|     0.5|
|              Boxing|   102| 187|  289|      0.35|    0.65|
|        Canoe Slalom|    41|  41|   82|       0.5|     0.5|
|        Canoe Sprint|   123| 126|  249|      0.49|    0.51|
|Cycling BMX Frees...|    10|   9|   19|      0.53|    0.47|
|  Cycling BMX Racing|  

In [0]:
medals_clean = medals
teams_clean = teams

# **_Storing the Tranformed Data in Transform Directory_**

In [0]:
athletes_clean_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Transformed Data/Athletes"
coaches_clean_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Transformed Data/Coaches"
teams_clean_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Transformed Data/Teams"
entriesGender_clean_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Transformed Data/EntriesGender"
medals_clean_path = "/Volumes/tokyoolympicsdb2/tokyoolympicschema/tokyo-olympic-data-volume/Transformed Data/Medals"

athletes_clean.write.mode("overwrite").option("header","true").csv(athletes_clean_path)
coaches_clean.write.mode("overwrite").option("header","true").csv(coaches_clean_path)
teams_clean.write.mode("overwrite").option("header","true").csv(teams_clean_path)
entriesGender_clean.write.mode("overwrite").option("header","true").csv(entriesGender_clean_path)
medals_clean.write.mode("overwrite").option("header","true").csv(medals_clean_path)

In [0]:
medals_clean.show()


+----+--------------------+----+------+------+-----+-------------+
|Rank|            Team/NOC|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   6|           Australia|  17|     7|    22|   46|            6|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|   9|             Germany|  10|    11|    16|   37|            8|
|  10|               Italy|  10|    10|    20|   40|            7|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          